# VEYL model

Supply the Git-ignored local/ archive and retained datasets and checkpoints before running these cells. Use the local uv environment for these checks. It needs NumPy and no PyTorch or Bun. The training guide explains the free Kaggle environment for Python inference and training.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "pyproject.toml").is_file(), "Open this notebook from the Veyl folder"
os.chdir(root)


def run(command):
    """Run a checked command and retain its output."""
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    print(result.stdout)


run([sys.executable, "-m", "train.check"])
run([sys.executable, "data/check.py"])
run([sys.executable, "eval/score.py", "--check"])


Training CPU checks passed; no training was started.

{
  "pinned_files": 251,
  "splits": {
    "broad/test.jsonl": {
      "rows": 1500,
      "unique_texts": 1500
    },
    "broad/train.jsonl": {
      "rows": 35485,
      "unique_texts": 35485
    },
    "broad/valid.jsonl": {
      "rows": 1500,
      "unique_texts": 1500
    },
    "tune/benign.train.jsonl": {
      "rows": 128,
      "unique_texts": 128
    },
    "tune/haiku.valid.jsonl": {
      "rows": 117,
      "unique_texts": 117
    },
    "tune/luna.train.jsonl": {
      "rows": 128,
      "unique_texts": 128
    },
    "tune/open.train.jsonl": {
      "rows": 4389,
      "unique_texts": 4389
    },
    "tune/open.valid.jsonl": {
      "rows": 250,
      "unique_texts": 250
    }
  },
  "scope": "File integrity and span bounds only. This does not verify label truth or semantic independence."
}

score self-check passed



## Retained deployment result

These measurements describe the INT8 WASM model. They do not score the FP32 Python loader. The local PIIMB figure is a four-task English mean, not an official leaderboard entry.


In [2]:
import json

report = json.loads(Path("local/reports/piimb.json").read_text())
print(json.dumps({"english_rows": report["rows"], "task_mean_f2": report["unweighted_task_f2_mean"], "micro": report["micro"]}, indent=2))


{
  "english_rows": 121409,
  "task_mean_f2": 0.8803466201782572,
  "micro": {
    "rows": 121409,
    "precision": 0.8424647099041429,
    "recall": 0.9537873134461334,
    "f1": 0.8946764060801914,
    "f2": 0.9292297795501342,
    "fpr": 0.032538621001200745,
    "support_chars": 1403013
  }
}


## Kaggle inference and training

Use Kaggle's existing PyTorch installation. Copy this folder and its retained inputs to a writable location. Run the following commands from that folder. The smoke check and FP32 inference can use a free CPU session. Full training requires a free GPU session.

```sh
python -m pip install uv
uv pip install --system -r pyproject.toml --extra train --extra onnx
python -m train.check --torch
python -m eval.model
python -m src.model 'Contact John Smith at john@example.com.'
python -m train.deep
python -m train.age
```

Deep trains the teacher for three epochs and distills the student for four. AGE uses the retained original Deep checkpoint and fixed source validation gates. A new Deep run is not promoted automatically. Both training commands refuse existing output folders. Exact cross-machine training weights are not promised.

## Optional deployment export

`uv run --locked --extra onnx python src/build.py --check` reproduces and verifies the selected graph from the saved AGE export. It does not retrain or quantize the FP32 loader again. The original fresh export recipe is in `train/common.py`. A new graph needs its own validation.
